# AutoML Exploration: SNe Ia Hubble Residual Prediction

Quick-and-dirty model comparison using AutoML libraries instead of manually benchmarking each algorithm.

**Libraries used:**
- **FLAML** (Microsoft) — works with Python 3.12, runs here
- **PyCaret** — nicer comparison tables, requires Python ≤3.11 (commented code for local use)

**Swap in your real data** at the clearly marked cell below.

In [2]:
# !pip install flaml[automl]
# !pip install pycaret  # only works on Python <=3.11

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

---
## 1. Load Data

### ⬇️ SWAP POINT: Replace the synthetic block with your real data ⬇️

In [3]:
import pandas as pd
df = pd.read_csv('/Users/pittsburghgraduatestudent/repos/myc21_first_paper/ZTF_DESI_ml_work/ZTF_DESI_data/ZTF_resid_cent_hostprop_with_x1_c.csv')
print(df.columns.tolist())
print(df.shape)

['Unnamed: 0', 'ztfname', 'redshift', 'redshift_err', 'source', 't0', 'x0', 'x1', 'c', 't0_err', 'x0_err', 'x1_err', 'c_err', 'cov_t0_x0', 'cov_t0_x1', 'cov_t0_c', 'cov_x0_x1', 'cov_x0_c', 'cov_x1_c', 'mwebv', 'mwr_v', 'mwebv_err', 'fitprob', 'ra', 'dec', 'sn_type', 'sub_type', 'lccoverage_flag', 'fitquality_flag', 'iau_name', 'frac_fitted', 'ra_host', 'dec_host', 'mass_err', 'mass', 'restframe_gz', 'restframe_gz_err', 'd_dlr', 'ANGSEP_ARCSEC', 'TARGETID', 'RA', 'DEC', 'DESI_TARGET', 'SPECTYPE', 'Z', 'ZWARN', 'ABS_DIFF', 'RESIDUAL', 'residual_centered', 'VDISP', 'LOGMSTAR', 'SFR', 'AGE', 'DN4000', 'ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_R', 'sigma_mu_meas2', 'sigma_mu_meas', 'mu_obs', 'mu_theory', 'residual']
(776, 61)


In [ ]:
# ============================================================
# OPTION B: Your real data
# ============================================================
# df = pd.read_csv('path/to/your/merged_ztf_desi.csv')
# # Adjust these to match your actual column names:
# feature_cols = ['x1', 'c', 'mB', 'log_mass', 'log_ssfr', 'metallicity', ...]
# target_col = 'hr'  # or 'hubble_residual', whatever you named it

target_col = 'hr'
feature_cols = [col for col in df.columns if col != target_col]

print(f"Dataset: {df.shape[0]} SNe, {len(feature_cols)} features")
df.describe().round(3)

In [ ]:
X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

---
## 2. FLAML AutoML — The One-Liner

This replaces the entire manual benchmarking notebook. FLAML will search over Random Forest, XGBoost, LightGBM, Extra Trees, and k-NN — tuning hyperparameters automatically within a time budget.

In [ ]:
automl = AutoML()

automl.fit(
    X_train, y_train,
    task='regression',
    metric='rmse',
    time_budget=120,       # seconds — bump to 300-600 for real data
    estimator_list=[
        'rf', 'xgboost', 'lgbm',
        'extra_tree', 'kneighbor'
    ],
    eval_method='cv',
    n_splits=5,
    seed=42,
    verbose=0,
)

In [ ]:
print("="*60)
print(f"Best model:          {automl.best_estimator}")
print(f"Best CV RMSE:        {-automl.best_loss:.5f} mag")
print(f"Best hyperparams:    {automl.best_config}")
print(f"Training time:       {automl.best_config_train_time:.1f}s")
print("="*60)

### Search history — what FLAML tried

In [ ]:
# Build a summary of all models FLAML tried
results = []
for trial in automl.results_.values() if hasattr(automl, 'results_') else []:
    if isinstance(trial, dict) and 'val_loss' in trial:
        results.append({
            'estimator': trial.get('learner', '?'),
            'rmse': -trial['val_loss'],
            'train_time': trial.get('train_time', 0)
        })

if results:
    results_df = pd.DataFrame(results)
    # Best per estimator
    best_per_model = results_df.groupby('estimator').agg(
        best_rmse=('rmse', 'min'),
        n_trials=('rmse', 'count'),
        total_time=('train_time', 'sum')
    ).sort_values('best_rmse')
    print(best_per_model.round(5).to_string())
else:
    # Fallback: just report the winner
    print("Detailed search history not available — see best model above.")

---
## 3. Evaluate on held-out test set

In [ ]:
y_pred = automl.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"Test RMSE:  {rmse:.5f} mag")
print(f"Test MAE:   {mae:.5f} mag")
print(f"Test R²:    {r2:.4f}")
print(f"Test σ(residual): {np.std(y_test - y_pred):.5f} mag")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Predicted vs true
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.4, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('True Hubble Residual (mag)')
ax.set_ylabel('Predicted Hubble Residual (mag)')
ax.set_title(f'Best model: {automl.best_estimator} | RMSE={rmse:.4f}')

# Residual distribution
ax = axes[1]
residuals = y_test - y_pred
ax.hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Residual (mag)')
ax.set_ylabel('Count')
ax.set_title(f'σ = {np.std(residuals):.4f} mag')

plt.tight_layout()
plt.show()

---
## 4. Feature Importance (from best model)

In [ ]:
model = automl.model.estimator

if hasattr(model, 'feature_importances_'):
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    imp.plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Feature Importance — {automl.best_estimator}')
    plt.tight_layout()
    plt.show()
elif hasattr(model, 'coef_'):
    imp = pd.Series(np.abs(model.coef_), index=feature_cols).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    imp.plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('|Coefficient|')
    ax.set_title(f'Feature Importance — {automl.best_estimator}')
    plt.tight_layout()
    plt.show()
else:
    print("Feature importances not directly available for this model type.")

---
## 5. PyCaret Version (run locally with Python ≤3.11)

PyCaret gives you the prettiest comparison table. Uncomment and run in your local env.

In [ ]:
# from pycaret.regression import *
#
# # Setup — handles preprocessing, CV, etc.
# reg = setup(
#     data=df,
#     target='hr',
#     session_id=42,
#     fold=5,
#     verbose=False
# )
#
# # This is the magic line — compares ~20 models in one call
# best = compare_models(sort='RMSE', n_select=3)
#
# # Tune the top model
# tuned = tune_model(best[0], optimize='RMSE')
#
# # Evaluate
# evaluate_model(tuned)      # interactive plots
# plot_model(tuned, plot='residuals')
# plot_model(tuned, plot='feature')

---
## Notes

- **Time budget**: 120s is fine for exploration. For real results, use 300–600s.
- **For the paper**: You'll still want the manual benchmarking notebook for transparency. But this is great for quickly checking whether you're missing a model family that outperforms your current picks.
- **Neural nets**: FLAML can include them (`estimator_list` + `'nn'`), but for ~1500 SNe with <10 features, tree methods will almost certainly dominate. NNs need much more data to justify their flexibility.